In [ ]:
from datetime import time

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import sys
sys.path.append('..')

In [ ]:
from tools.geometry import generate_detector
import jax.numpy as jnp
default_json_filename='../config/cyl_geom_config.json'
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)

In [ ]:
from tools.propagate import create_photon_propagator
from tools.geometry import generate_detector
from tools.utils import load_single_event, save_single_event, print_particle_params, print_detector_params, generate_random_point_inside_cylinder, generate_random_params
from tools.losses import compute_loss_with_time
from tools.simulation import setup_event_simulator

import jax.numpy as jnp
from functools import partial
import pickle
from tqdm import tqdm
import jax
from jax import grad, jit, vmap, value_and_grad
import time
from tools.losses import compute_simplified_loss, compute_simple_loss, compute_softmin_loss

# create the ../plots folder if it does not exist
import os
if not os.path.exists('../plots'):
    os.makedirs('../plots')
    print('Created ../plots folder')

default_json_filename = '../config/cyl_geom_config.json'

detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)
Nphot = 3_000_000

K=2

simulate_event = setup_event_simulator(default_json_filename, Nphot, temperature=0.05, K=K, is_calibration=False, max_detectors_per_cell=10)

# Generate and save a single event
key_id = 43
key = jax.random.PRNGKey(key_id)

trk_params = generate_random_params(key)
detector_params = (
    jnp.array(3),           # scatter_length
    jnp.array(0.00),         # reflection_rate
    jnp.array(999.),         # absorption_length
    jnp.array(0.001)         # gumbel_softmax_temp
)

single_event_data = jax.lax.stop_gradient(simulate_event(trk_params, detector_params, key))
print_particle_params(trk_params)

save_single_event(single_event_data, trk_params, detector_params, filename='../events/test_event_data.h5', calibration_mode=False)
print('Done')

In [ ]:
from tools.utils import load_single_event
event_location = '../events/test_event_data.h5'
loaded_trk_params, loaded_detector_params, loaded_indices, loaded_charges, loaded_times= load_single_event(event_location, 9946, calibration_mode=False)

In [ ]:
from tools.visualization import create_detector_display
detector_display = create_detector_display(default_json_filename)
detector_display(loaded_indices, loaded_charges, loaded_times, file_name='../plots/event_display_charge.png', plot_time=False, log_scale=True)
detector_display(loaded_indices, loaded_charges, loaded_times, file_name='../plots/event_display_time.png', plot_time=True)

In [ ]:
detector.visualize_event_data_plotly(loaded_indices, loaded_charges, loaded_times, show_all_detectors=True, log_scale=True, show_colorbar=False, dark_theme=False)
